# AE + Contrastive (SimCLR) + MNA ile Açı+Hız Eğitimi

**Amaç:** Önceden çıkarılmış XY `.npy` dosyalarını kullanarak:
- Üst beden açılar (6 kanal) + **açısal hızlar** (6 kanal) ⇒ toplam **12 özellik** çıkar,
- **Pencereleme** (W=30, stride=15) ve **z-score** normalizasyon uygula,
- **Encoder: 1D-CNN → GAP → FC**, **Decoder: Multi-head (MNA)**, + **SimCLR (NT-Xent)** ile eğit,
- (Opsiyonel) başarı yüzdesi ve latent çıkarım.

 Neden **1D-CNN**? Zaman eksenindeki lokal desenleri hızlı ve stabil yakalar.  
 Neden **GAP**? Zaman boyutunu özetleyip parametreyi azaltır, uzunluğa karşı dayanıklıdır.


## 1.Kurulum, yollar, hiperparametreler

In [12]:
import os, re, glob, random
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ========== YOL TANIMLARI ==========
BASE = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"

DIR_TR        = os.path.join(BASE, "npy_aci_cikti")         # Eğitim veri seti (açı)
DIR_TE        = os.path.join(BASE, "npy_aci_test_cikti")    # Test veri seti (açı)
OUTDIR        = os.path.join(BASE, "ae_mna_pose_angles_out")  # Kayıt klasörü

os.makedirs(OUTDIR, exist_ok=True)  # Kayıt klasörü yoksa oluştur

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# --- .npy arayıcı (hem .npy hem .NPY yakalar) ---
def _list_npy_case_insensitive(folder: str):
    return (glob.glob(os.path.join(folder, "*.npy")) +
            glob.glob(os.path.join(folder, "*.[nN][pP][yY]")))

print("BASE:", BASE)
print("Başlangıç DIR_TR:", DIR_TR)

# Eğer DIR_TR içinde .npy yoksa, BASE altında en çok .npy olan klasörü otomatik seç
cand_dir = DIR_TR
if (not os.path.isdir(cand_dir)) or (len(_list_npy_case_insensitive(cand_dir)) == 0):
    counts = {}
    for p in Path(BASE).rglob("*.[nN][pP][yY]"):
        d = str(p.parent)
        counts[d] = counts.get(d, 0) + 1
    assert counts, "BASE altında hiç .npy bulunamadı; BASE yolunu doğru klasöre ayarla."

    # 'test' içermeyen klasörleri tercih et (eğitim için)
    chosen = sorted(counts.items(), key=lambda x: -x[1])
    picked = None
    for d, c in chosen:
        if "test" not in d.lower():
            picked = d
            break
    if picked is None:
        picked = chosen[0][0]

    cand_dir = picked
    print("Uyarı: Başlangıç DIR_TR içinde .npy yoktu, otomatik seçildi ->", cand_dir)

# Son karar
DIR_TR = cand_dir
print("Kullanılacak DIR_TR:", DIR_TR, "| .npy sayısı:", len(_list_npy_case_insensitive(DIR_TR)))
print("DIR_TE:", DIR_TE, "| mevcut mu:", os.path.isdir(DIR_TE))

# ==== Hiperparametreler ====
WINDOW = 30
STRIDE = 15
BATCH_SIZE = 128
EPOCHS = 70
LR = 1e-3
LATENT_DIM = 128

# Contrastive (SimCLR)
LAMBDA_CONTR = 0.5
TEMP = 0.5

# MNA (komşu pencere rekonstrüksiyon)
OFFS = [-1, 1]
BETA_MNA = 0.3

VAL_SPLIT = 0.2
PATIENCE = 10


BASE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video
Başlangıç DIR_TR: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\npy_aci_cikti
Kullanılacak DIR_TR: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\npy_aci_cikti | .npy sayısı: 120
DIR_TE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\npy_aci_test_cikti | mevcut mu: True


## 2. Yardımcı fonksiyonlar: dosya listeleme, pencereleme


In [13]:
def list_npy(folder: str) -> List[str]:
    files = glob.glob(os.path.join(folder, "*.npy"))
    def key_fn(p):
        m = re.findall(r"(\d+)", os.path.basename(p))
        return int(m[0]) if m else 10**9
    return sorted(files, key=key_fn)

def extract_id(p: str) -> int:
    m = re.findall(r"(\d+)", os.path.basename(p))
    return int(m[0]) if m else -1

def windowize(seq: np.ndarray, W: int, S: int) -> np.ndarray:
    T, F = seq.shape
    out = []
    for s in range(0, max(T - W + 1, 0), S):
        out.append(seq[s:s+W])
    return np.asarray(out, dtype=np.float32)


## 3. Üst beden 17 nokta → açı ve açısal hız üretimi
- Dirsek: ∠(omuz–dirsek–bilek)
- Omuz: ∠(kalça–omuz–dirsek)
- Gövde yaw: kalça hattı → omuz hattı signed açı
- Gövde eğimi: (kalça orta → omuz orta) vektörü ile dikey arasındaki açı
Toplam **6 açı**, + frame farkı ile **6 hız** ⇒ **12 kanal**.


In [14]:
# --- COCO-17 ve MediaPipe-33 için eklem eşlemeleri (üst beden) ---
COCO17 = dict(LShoulder=5, RShoulder=6, LElbow=7, RElbow=8, LWrist=9,  RWrist=10, LHip=11, RHip=12)
MP33   = dict(LShoulder=11, RShoulder=12, LElbow=13, RElbow=14, LWrist=15, RWrist=16, LHip=23, RHip=24)

def pick_joints(Fxy: int):
    """XY sütun sayısına (Fxy) göre 17/33 noktalı eklem eşlemesini seç."""
    pts = Fxy // 2
    if pts == 17: return COCO17
    if pts == 33: return MP33
    return None  # XY değilse None (örneğin zaten özellik serisi)

# ---- yardımcı fonksiyonlar ----

def angle_wrap(a):  # [-pi, pi] aralığına sarar
    return (a + np.pi) % (2*np.pi) - np.pi

def angle_diff(a):  # ardışık açı farkları (a[1]-a[0], ...), wrap-aware şekilde
    da = np.diff(a, axis=0, prepend=a[[0], :])      # ilk frame'i kendisiyle farkla başlat
    return angle_wrap(da)                           # sarma duyarlılığı eklenir

def xy_to_points(frame_xy: np.ndarray):  # [Fxy] vektörünü [N,2] pozisyon dizisine çevir
    return frame_xy.reshape(-1, 2)

def vec(p1, p2):  # vektör: p2 - p1
    return p2 - p1

def angle_between(u, v):  # 2D düzlemde yönlü açı (radyan) → atan2 ile
    ang1 = np.arctan2(u[...,1], u[...,0])  # u'nun açısı
    ang2 = np.arctan2(v[...,1], v[...,0])  # v'nin açısı
    return angle_wrap(ang2 - ang1)         # farkı [-pi, pi]'ye sar

def angle_at(A, B, C):   # ∠ABC → B'deki iç açı (0..π)
    BA = A - B; BC = C - B                        # vektörler
    num = (BA * BC).sum(axis=-1)                 # iç çarpım
    den = (np.linalg.norm(BA,axis=-1) * np.linalg.norm(BC,axis=-1) + 1e-8)
    cosv = np.clip(num/den, -1.0, 1.0)           # cos(θ) hesapla
    return np.arccos(cosv)                       # açıya çevir

# ---- XY → 6 açılık açı dizisi çıkarımı ----
def compute_angles_xy(seq_xy: np.ndarray) -> np.ndarray:
    """
    XY koordinat verisini 6 temel açıya dönüştürür.
    Girdi: (T, Fxy) → Çıktı: (T, 6)
    Açılar sırasıyla:
      - LE: Sol dirsek açısı
      - RE: Sağ dirsek açısı
      - LS: Sol omuz (gövde-kol)
      - RS: Sağ omuz (gövde-kol)
      - yaw: Gövde yönü (kalça–omuz ekseni)
      - lean: Gövde eğimi (dikey eksene göre)
    """
    T, Fxy = seq_xy.shape
    J = pick_joints(Fxy)  # COCO-17 veya MP-33 eklem eşlemesi seç
    assert (Fxy % 2 == 0) and (J is not None), "XY formatı bekleniyordu (17 veya 33 nokta)."

    ang = np.zeros((T, 6), dtype=np.float32)
    for t in range(T):
        P  = xy_to_points(seq_xy[t])         # (N, 2) formatına çevir
        LS = P[J["LShoulder"]]; RS = P[J["RShoulder"]]
        LE = P[J["LElbow"]];    RE = P[J["RElbow"]]
        LW = P[J["LWrist"]];    RW = P[J["RWrist"]]
        LH = P[J["LHip"]];      RH = P[J["RHip"]]

        # Dirsek açıları: Shoulder–Elbow–Wrist
        a_le = angle_at(LS, LE, LW)
        a_re = angle_at(RS, RE, RW)

        # Omuz açıları: Hip–Shoulder–Elbow
        a_ls = angle_at(LH, LS, LE)
        a_rs = angle_at(RH, RS, RE)

        # Gövde yaw: kalça çizgisi ile omuz çizgisi arasındaki yönlü açı
        v_hp = vec(LH, RH); v_sh = vec(LS, RS)
        yaw  = angle_between(v_hp, v_sh)

        # Gövde eğimi (lean): dikey ile omuz–kalça orta noktaları arasındaki vektörün açısı
        mid_hip = (LH + RH)/2.0; mid_sh = (LS + RS)/2.0
        v_tr = vec(mid_hip, mid_sh)
        v_axis = np.array([0.0, -1.0], np.float32)  # yukarı dikey
        lean = angle_between(v_axis, v_tr)

        ang[t] = [a_le, a_re, a_ls, a_rs, yaw, lean]  # sıralı kayıt

    return ang  # (T,6)

# ---- Açılardan sin, cos ve açısal hız çıkart (toplam 18 kanal) ----
def build_features_from_xy(seq_xy: np.ndarray) -> np.ndarray:
    """
    Girdi:
      seq_xy : (T, Fxy) → XY koordinat dizisi (ör: 17 nokta → 34 kolon)
    Çıktı:
      feats  : (T, 18) = [sin(a 6), cos(a 6), d(a) 6]

    Açıklamalar:
      - sin/cos dönüşümü, açıların +π/-π sıçramasını engeller (döngüsel uzayda daha sağlıklı öğrenilir)
      - d(a), wrap-aware ardışık farktır (sıçrama yerine doğru yönü verir)
      - Toplamda her frame için 18 özellik çıkmış olur
    """
    ang  = compute_angles_xy(seq_xy).astype(np.float32)   # (T, 6) → Açılar
    dang = angle_diff(ang).astype(np.float32)             # (T, 6) → Açısal hız
    s = np.sin(ang).astype(np.float32)                    # (T, 6) → sin bileşeni
    c = np.cos(ang).astype(np.float32)                    # (T, 6) → cos bileşeni
    feats = np.concatenate([s, c, dang], axis=1).astype(np.float32)  # (T, 18)
    return feats


In [15]:
def detect_format(arr: np.ndarray) -> str:
    """
    Girdi olan dizinin formatını tespit eder:
      - XY koordinatları mı? (17 veya 33 nokta x 2 = 34 veya 66 sütun)
      - Yoksa doğrudan özellik dizisi mi?
    
    Girdi:
        arr: (T, F) boyutunda NumPy dizisi
            T = zaman/frame sayısı, F = özellik/koordinat sayısı
    
    Dönüş:
        'xy'   → XY formatı ise (MediaPipe veya COCO)
        'feat' → Özellik (feature) formatı ise (örneğin açılar)
    """
    if arr.ndim == 2 and arr.shape[1] % 2 == 0 and pick_joints(arr.shape[1]) is not None:
        return "xy"  # Eğer 2D, çift sayıda sütun ve XY eklemleri varsa
    return "feat"    # Aksi durumda doğrudan feature dizisi kabul edilir


def build_features_from_numeric(seq: np.ndarray) -> np.ndarray:
    """
    XY olmayan, doğrudan açı/özellik serileri için:
    Ham veriye ek olarak:
      - 1. fark (hız) → d1
      - 2. fark (ivme) → d2
    Eklenerek toplamda 3 * F boyutunda bir özellik vektörü oluşturulur.
    
    Girdi:
        seq: (T, F) → T frame, F adet özellik (ör. açı)
    
    Dönüş:
        feats: (T, 3*F) = [ham, hız, ivme]
    
    Notlar:
      - `prepend=seq[[0], :]` → İlk frame'in kendisiyle fark alınmasını sağlar (başta sıfır kayıp).
      - Bu işlem özellikle iskelet açılarında zaman içindeki değişimi anlamlı hale getirir.
    """
    d1 = np.diff(seq, axis=0, prepend=seq[[0], :])   # 1. fark → velocity (hız)
    d2 = np.diff(d1, axis=0, prepend=d1[[0], :])     # 2. fark → acceleration (ivme)
    return np.concatenate([seq, d1, d2], axis=1).astype(np.float32)


## 4. Dataset: pencereleme + z-score + SimCLR augment + MNA komşuları

XY → açı özellikleri: Eklemler arası ilişkileri yakalar, kamera ölçeğinden bağımsızdır; sin/cos sarılma sorununu çözer.

[ham, 1.fark, 2.fark]: Zaman dinamiğini (hız/ivme benzeri) kodlar; daha ayrıştırıcı temsil verir.

Z-score normalizasyonu: Kanallar arası ölçek farkını giderir; eğitim stabilitesini artırır.

Augment seti: Jitter/scale/mask/shift/time-warp; contrastive için çeşitlilik ve genelleme sağlar.

MNA komşuları + mask: Sınır durumlarında kaybı kirletmemek için pad’lenmiş komşuları maskeleriz.

(C, L) formatı: PyTorch 1D-CNN/Conv1d giriş düzeniyle uyumlu olacak şekilde transpose edilir.

In [16]:
# Dataset sınıfı (GÜNCEL)

class PoseWindowDataset(Dataset):
    """
    .npy dosyalarından pencere (window) üretir ve eğitim için gerekli tensörleri döndürür.

    Girdi formatı iki şekilde olabilir:
      1) XY koordinat dizisi (ör. MediaPipe 17/33 nokta)      -> önce açı-özelliklerine dönüştürürüz
         - 6 açı için [sin, cos] + açısal hız (derivative)     -> kanal sayısı C = 18
      2) Doğrudan özellik serisi (angle/features)             -> [ham, 1.fark, 2.fark]   -> C = 3 * F

    Dönüş tensörlerinin şekilleri:
      x_clean, x1, x2 : (C, L)         -> merkez pencere (temiz) + SimCLR için 2 görünüm
      neighs          : (K, C, L)      -> MNA için komşu pencereler (K = len(OFFS))
      mask            : (K,)           -> komşu pencerelerden hangileri "geçerli" (0/1)
    """
    def __init__(self, files: List[str], window=30, stride=15,
                 fit_stats=True, stats=None):
        # Dosya listesi ve pencereleme parametreleri
        self.files = files
        self.window = window   # Her örneğin uzunluğu (frame sayısı)
        self.stride = stride   # Pencere kaydırma miktarı (örtüşme kontrolü)

        # all_wins : Bütün dosyalardan üretilen pencereleri tek havuzda toplarız (N, W, C)
        # meta     : Global indeks -> (fid, pos) eşleşmesi; geri izleme için
        # base     : Her dosyanın global start indeksini tutar (hızlı erişim için)
        # lenf     : Her dosyanın toplam pencere sayısı
        self.all_wins = []   # [N, W, C]
        self.meta = []       # global index -> (fid, pos)
        self.base = {}       # fid -> global start
        self.lenf = {}       # fid -> #wins

        # gptr: all_wins'e eklenen her pencere için artan global sayaç
        gptr = 0
        for fp in self.files:
            # Her dosyayı yükle: (T, F) -> T: zaman (frame), F: özellik/koordinat sayısı
            arr = np.load(fp).astype(np.float32)

            # --- FORMAT ALGILAMA ---
            # Neden? XY koordinatlarıyla ham özellik serilerini farklı işliyoruz.
            fmt = detect_format(arr)  # 'xy' veya 'feat'
            if fmt == "xy":
                # XY'den model için daha anlamlı ve ölçü biriminden bağımsız açısal özellikler üretiriz.
                # [sin, cos] kullanımı, açıların 2π sarılma (wrap-around) problemine karşı stabildir.
                feats = build_features_from_xy(arr)       # (T, 18)
            else:
                # Zaten özellik serisiyse, dinamiği yakalamak için fark özellikleri ekleriz.
                # (ham + 1. fark + 2. fark) -> hızlanma/ivme benzeri türev bilgileri
                feats = build_features_from_numeric(arr)  # (T, 3*F)

            # --- PENCERELEME ---
            # Zaman serisini kayan pencerelere böleriz. (nW, W, C)
            # Window: modelin bir anda gördüğü yerel bağlam; Stride: örnek sayısı ve örtüşme kontrolü
            wins = windowize(feats, self.window, self.stride)  # (nW, W, C)

            # Dosya kimliği (raporlama/geri izleme için)
            fid = extract_id(fp)
            self.base[fid] = gptr               # Bu dosyanın pencereleri globalde nereden başlıyor?
            self.lenf[fid] = len(wins)          # Bu dosyadan kaç pencere çıktı?

            # Pencereleri global havuza ekle ve meta haritasını doldur
            for pos in range(len(wins)):
                self.all_wins.append(wins[pos])   # (W, C)
                self.meta.append((fid, pos))      # global_idx -> (fid, dosya içi pencere konumu)
                gptr += 1

        # Artık tüm pencereler bir arada: (N, W, C)
        self.all_wins = np.asarray(self.all_wins, dtype=np.float32)
        if self.all_wins.size == 0:
            # Kullanıcı hatalarını hızlı yakalamak için açık uyarı.
            raise RuntimeError("Pencere üretilemedi. WINDOW/STRIDE veya veri uzunluğunu kontrol et.")

        # --- Z-SCORE NORMALİZASYON ---
        # Neden? Kanal ölçeklerini hizalamak ve eğitimde kararlılığı artırmak için.
        # fit_stats=True ve stats=None ise istatistikleri bu dataset üzerinden hesaplarız.
        # stats verildiyse (ör. train'den gelen), aynısını uygularız -> test/val tutarlılığı.
        if stats is None and fit_stats:
            # (0,1) eksenleri: zaman ve örnek ortalaması -> kanal başına mean/std
            self.mean = self.all_wins.mean(axis=(0, 1), keepdims=True)
            self.std  = self.all_wins.std(axis=(0, 1), keepdims=True) + 1e-8  # sıfıra bölmeyi engelle
        else:
            self.mean, self.std = stats

        # Veriyi normalize et: (N, W, C)
        self.all_wins = (self.all_wins - self.mean) / self.std

    def get_stats(self):
        # Eğitimde hesaplanan (veya dışarıdan verilen) z-score istatistiklerini geriye döndür.
        # Model kaydına bu istatistikleri ekleyip inference/val/test’te aynısını kullanırız.
        return (self.mean, self.std)

    # ------ SimCLR augment'leri ------
    # Neden augment? Contrastive öğrenmede iki farklı "görünüm" üretip, modelin
    # aynı pencereden gelen varyasyonları birbirine yakın, farklı pencereleri uzak
    # konumlandırmasını sağlarız.
    def _aug_jitter(self, x, sigma=0.02):
        # Küçük Gauss gürültüsü: sensör/ölçüm hatalarını taklit eder.
        return x + np.random.normal(0, sigma, size=x.shape).astype(np.float32)

    def _aug_scale(self, x, smin=0.85, smax=1.15):
        # Genlik ölçekleme: açı/özelliklerin mutlak büyüklüğündeki farklılıkları robuslaştırır.
        s = np.random.uniform(smin, smax)
        return (x * s).astype(np.float32)

    def _aug_time_mask(self, x, max_len=6):
        # Zaman ekseninde kısa bir bölümü sıfırlama (SpecAugment benzeri);
        # Kısmi kayıp bilgiye karşı dayanıklılık kazandırır.
        x = x.copy(); L = x.shape[0]
        m = np.random.randint(1, max_len+1)
        s = np.random.randint(0, max(1, L - m + 1))
        x[s:s+m] = 0.0
        return x

    def _aug_shift(self, x, max_shift=4):
        # Zaman kaydırma: penceredeki olayın başlangıcındaki küçük ötelemelere karşı esneklik.
        sh = np.random.randint(-max_shift, max_shift+1)
        return np.roll(x, sh, axis=0).astype(np.float32)

    def _aug_time_warp(self, x, warp_min=0.9, warp_max=1.1):
        """
        Zaman ölçekleme (hızlandır/yavaşlat) – L sabit kalır.
        Basit lineer enterpolasyon kullanırız:
          1) Oranı kadar örnekleyip yeni uzunlukta sinyal üret,
          2) Yeniden orijinal L'e ölçekle.
        """
        L, C = x.shape
        rate = np.random.uniform(warp_min, warp_max)

        # 1) Yeni uzunluğa örnekle
        new_L = max(2, int(round(L * rate)))
        t_src = np.linspace(0, 1, L)
        t_new = np.linspace(0, 1, new_L)
        x_w = np.vstack([np.interp(t_new, t_src, x[:, k]) for k in range(C)]).T  # (new_L, C)

        # 2) Geri orijinal L'e ölçekle
        t_back = np.linspace(0, 1, L)
        x_out = np.vstack([np.interp(t_back, np.linspace(0, 1, new_L), x_w[:, k]) for k in range(C)]).T
        return x_out.astype(np.float32)

    def _apply_augs(self, x):
        # Her çağrıda rastgele 2 farklı augment seçiyoruz (replace=False).
        # Bu çeşitlilik, contrastive görev için daha zengin "görünüm" uzayı sağlar.
        funcs = [self._aug_jitter, self._aug_scale, self._aug_time_mask, self._aug_shift, self._aug_time_warp]
        fs = np.random.choice(funcs, size=2, replace=False)
        y = x
        for f in fs:
            y = f(y)
        return y.astype(np.float32)

    def __len__(self):
        # Dataset uzunluğu: toplam pencere sayısı
        return len(self.all_wins)

    def __getitem__(self, idx):
        # 1) Merkez pencere (temiz) — (W, C)
        x = self.all_wins[idx]
        fid, pos = self.meta[idx]  # Hangi dosyadan ve o dosyada kaçıncı pencere?

        # 2) MNA komşuları:
        # OFFS global listesinde tanımlı ofsetlere göre, komşu pencereleri topluyoruz.
        # Sınır dışına taşıyorsa (baş/son), sıfır penceresi koyup mask=0 yapıyoruz.
        neighs, mask = [], []
        for d in OFFS:
            q = pos + d
            if 0 <= q < self.lenf[fid]:
                g = self.base[fid] + q
                neighs.append(self.all_wins[g])   # (W, C)
                mask.append(1.0)                  # geçerli komşu
            else:
                neighs.append(np.zeros_like(x))   # pad
                mask.append(0.0)                  # geçersiz komşu (kayıp hesapta dışlanacak)
        neighs = np.stack(neighs, axis=0)         # (K, W, C)
        mask   = np.asarray(mask, np.float32)     # (K,)

        # 3) SimCLR için iki görünüm üret (aynı merkez pencereden)
        x1 = self._apply_augs(x)
        x2 = self._apply_augs(x)

        # 4) PyTorch kanal düzenine çevir: (C, L)
        #  Not: Eğitim döngüsünde (B, C, L) bekleniyor; DataLoader batch'leyince B eklenecek.
        x_clean = torch.from_numpy(x.T.copy())                        # (C, W)
        x1      = torch.from_numpy(x1.T.copy())
        x2      = torch.from_numpy(x2.T.copy())
        neighs  = torch.from_numpy(neighs.transpose(0, 2, 1).copy())  # (K, C, W)
        mask    = torch.from_numpy(mask)                              # (K,)

        # Dönüş sırası, eğitim döngüsü ile birebir uyumlu olmalı:
        #   x_clean, x1, x2 -> model merkez/contrastive için
        #   neighs, mask    -> komşu rekonstrüksiyon ve maskeli kayıp için
        return x_clean, x1, x2, neighs, mask


In [17]:
ds_tr = PoseWindowDataset(train_files, WINDOW, STRIDE, fit_stats=True)
print("Train windows:", ds_tr.all_wins.shape)   # (N, W, C)
print("CIN:", ds_tr.all_wins.shape[-1])


Train windows: (574, 30, 54)
CIN: 54


## 5. Model: Encoder (1D-CNN → GAP → FC) + Decoder (center + komşular)
- 1D-CNN zaman eksenindeki lokal paternleri yakalar,
- GAP zaman boyunca özet çıkarır,
- FC latent vektörü üretir,
- Decoder, merkez pencere ve her komşu için ayrı head kullanır (MNA).


Conv1d: Zaman serisi verilerde lokal ilişkileri (hareket akışı) yakalar.

ReLU: Hesaplama ucuz, gradyan kaybını engeller.

AdaptiveAvgPool1d: Pencere uzunluğundan bağımsız sabit boyut elde etmek için.

Latent (fc_mu): Ham veriyi düşük boyutlu temsil (özellik vektörü) haline getirir.

Projection Head: SimCLR/contrastive öğrenmede z’yi daha ayrıştırıcı hale getirir.

Ayrı Decoder’lar: Hem merkez hem farklı zaman ofsetlerini yeniden kurabilmek için


In [18]:
class MetricAE_MNA(nn.Module):
    """
    Autoencoder (AE) + MNA (Masked Neighbor Autoencoding) + SimCLR (contrastive)
      - Encoder: 1D-CNN -> GAP -> FC (latent z)
      - Projection head: MLP (g(z)) -> contrastive kaybı bununla hesaplanır
      - Decoder'lar: merkez + her komşu ofset için ayrı linear katman
    """
    def __init__(self, in_channels: int, seq_len: int, latent_dim: int, offsets: list[int]):
        super().__init__()
        self.in_channels = in_channels    # Girdi kanal sayısı (ör: koordinat + açı boyutları)
        self.seq_len = seq_len            # Pencere uzunluğu (frame sayısı)
        # Ofsetleri string saklıyoruz çünkü ModuleDict anahtarları string olmalı
        self.offsets = [str(d) for d in offsets]

        # --- Encoder ---
        # Amaç: (B, C, L) girişi daha küçük latent temsil z’ye sıkıştırmak
        # Burada Conv1d seçilmesinin sebebi: zaman serisi verilerinde (frame dizileri) lokal paternleri öğrenmek için uygun
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, padding=2), nn.ReLU(),   # İlk konvolüsyon → düşük seviye özellikler
            nn.Conv1d(64, 128,      kernel_size=5, padding=2), nn.ReLU(),      # Daha derin özellikler
            nn.Conv1d(128, 128,     kernel_size=3, padding=1), nn.ReLU(),      # İnce detayları yakalama
            nn.AdaptiveAvgPool1d(1),    # GAP (Global Average Pooling): zaman boyutunu özetler
            nn.Flatten()                # (B, 128, 1) → (B, 128)
        )
        self.fc_mu = nn.Linear(128, latent_dim)  # Latent z (B,D) → temsil vektörü
        # Neden Linear? CNN çıktısını sabit boyutlu latent vektöre indirmek için en basit ve etkili yol.

        # --- Projection head (SimCLR) ---
        # Contrastive öğrenmede z yerine g(z) kullanılır çünkü latent z daha genel,
        # g(z) ise benzerlik ayırma görevine özel optimize edilir.
        self.proj_head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim), nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)  # Çıkış boyutu latent_dim (B,D)
        )

        # --- Decoder'lar ---
        # Merkez pencere rekonstrüksiyonu için ayrı decoder
        self.dec_center = nn.Linear(latent_dim, in_channels * seq_len)
        # Komşu pencereler için her ofsete özel ayrı Linear decoder
        # Neden ayrı? Her ofset farklı zaman kaymasını temsil eder, aynı decoder paylaşılamaz.
        self.dec_neighs = nn.ModuleDict({
            k: nn.Linear(latent_dim, in_channels * seq_len) for k in self.offsets
        })

    # ---------- yardımcılar ----------
    def _reshape(self, y):
        # Linear çıkış (B, C*L) → geri (B, C, L) boyutuna dönüştürülür
        return y.view(-1, self.in_channels, self.seq_len)

    # ---------- ileri yayılım parçaları ----------
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B,C,L) → z: (B,D)"""
        h = self.encoder(x)    # CNN + GAP ile özellik çıkarımı
        z = self.fc_mu(h)      # Latent vektör
        return z

    def project(self, z: torch.Tensor) -> torch.Tensor:
        """z: (B,D) → g(z): (B,D), contrastive loss bu çıktıyla hesaplanır"""
        return self.proj_head(z)

    def decode_center(self, z: torch.Tensor) -> torch.Tensor:
        """z → merkez pencere rekonstrüksiyonu (B,C,L)"""
        return self._reshape(self.dec_center(z))

    def decode_neighs(self, z: torch.Tensor) -> dict:
        """z → komşu pencerelerin rekonstrüksiyonları dict[str]→(B,C,L)"""
        return {k: self._reshape(self.dec_neighs[k](z)) for k in self.offsets}

    def forward(self, x: torch.Tensor):
        """
        Dönüş değerleri:
          y0 : (B,C,L)  → merkez rekonstrüksiyon
          yN : dict[str]→ her komşu için rekonstrüksiyon
          z  : (B,D)    → latent vektör
          p  : (B,D)    → projection head çıktısı (contrastive için)
        """
        z = self.encode(x)             # Latent vektör
        p = self.project(z)            # Contrastive head çıktısı
        y0 = self.decode_center(z)     # Merkez pencere tahmini
        yN = self.decode_neighs(z)     # Komşu pencere tahminleri
        return y0, yN, z, p


## 6. Contrastive loss (SimCLR / NT-Xent)
SimCLR gibi contrastive learning yöntemlerinde iki augment edilmiş örneği (z1 ve z2) latent uzayda birbirine yaklaştırmaya, diğer tüm örneklerden uzaklaştırmaya çalışır.

In [19]:
def nt_xent(z1, z2, temp=0.2):
    """
    SimCLR için NT-Xent (Normalized Temperature-scaled Cross Entropy) loss hesaplar.
    İki farklı görünüm (augment edilmiş) vektörlerini (z1, z2) karşılaştırır.

    Girdi:
        z1, z2 : (B, D) → augment edilmiş iki görünümden gelen latent vektörler
        temp  : float → sıcaklık değeri (softmax keskinliğini ayarlar)
    
    Dönüş:
        Contrastive loss değeri (float)
    """

    # --- 1. Normalize: z1 ve z2'yi birim vektör yap (cosine benzerliği için önemli)
    z1 = nn.functional.normalize(z1, dim=1)
    z2 = nn.functional.normalize(z2, dim=1)

    B = z1.size(0)  # Batch boyutu

    # --- 2. Tüm örnekleri birleştir: (2B, D) boyutlu matris elde edilir
    reps = torch.cat([z1, z2], dim=0)          

    # --- 3. Cosine benzerlik matrisi: (2B, 2B)
    #    Her vektörün tüm diğerleriyle benzerliği hesaplanır
    sim  = torch.matmul(reps, reps.T) / temp   # sıcaklıkla ölçeklenir

    # --- 4. Diagonal maskelenir: bir örnek kendisiyle eşleşemez
    mask = torch.eye(2*B, dtype=torch.bool, device=reps.device)
    sim  = sim.masked_fill(mask, -1e9)  # kendi kendine benzerliği dışla

    # --- 5. Doğru eşleşmeleri gösteren hedef etiket vektörü hazırlanır
    #    z1[i] için pozitif örnek z2[i], z2[i] için z1[i] → (sıralı eşleme)
    targets = torch.cat([torch.arange(B, 2*B), torch.arange(0, B)]).to(reps.device)

    # --- 6. Cross-Entropy Loss: doğru eşleştirilen çiftlerin similarity değeri yüksek olsun
    return nn.CrossEntropyLoss()(sim, targets)


## 7. Data split + DataLoader


In [20]:
# .npy dosyalarının şekillerini örnekle (XY mi kontrol)
files = list_npy(DIR_TR)
print("Toplam dosya:", len(files))
for fp in files[:5]:
    arr = np.load(fp, mmap_mode="r")
    print(os.path.basename(fp), "->", arr.shape)

# Beklenti:
#  - 17 nokta XY ise: (T, 34)
#  - MediaPipe 33 nokta XY ise: (T, 66)
#  Not: Son boyut çift olmalı (XY)


Toplam dosya: 60
1_ang.npy -> (259, 3)
2_ang.npy -> (273, 3)
3_ang.npy -> (173, 3)
4_ang.npy -> (231, 3)
5_ang.npy -> (197, 3)


In [21]:
# --- 1. Eğitim klasöründeki .npy dosyalarını listele ---
all_files = list_npy(DIR_TR)  # Eğitim için kullanılacak tüm .npy dosyalarını al

# Eğitim seti boş olamaz; kontrol et
assert len(all_files) > 0, "Eğitim klasöründe .npy yok."

# --- 2. Eğitim–validasyon ayırımı (ör: %20 validasyon) ---
n_val = max(1, int(len(all_files) * VAL_SPLIT))  # En az 1 validasyon dosyası
val_files   = all_files[-n_val:]                 # Son n dosya validasyon
train_files = all_files[:-n_val]                 # Kalanlar eğitim

# --- 3. Dataset nesneleri oluştur (pencereleme + normalizasyon dahil) ---

# Eğitim verisi (fit_stats=True → mean/std burada hesaplanır)
ds_tr = PoseWindowDataset(train_files, WINDOW, STRIDE, fit_stats=True)

# Hesaplanan z-score istatistikleri (eğitim verisinin ortalama ve std’si)
stats = ds_tr.get_stats()

# Validasyon verisi (eğitim istatistikleriyle normalize edilir)
ds_va = PoseWindowDataset(val_files, WINDOW, STRIDE, fit_stats=False, stats=stats)

# --- 4. Giriş boyutu ve pencere uzunluğu ayarları ---
CIN   = ds_tr.all_wins.shape[-1]  # Kanal sayısı (feature sayısı) örn: 12 veya 18
LSEQ  = WINDOW                    # Pencere uzunluğu (zaman boyutu)

# --- 5. DataLoader oluştur (mini-batch ve shuffle ayarları) ---

# Eğitim loader: batch + shuffle + drop_last (sabit boyut için)
dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)

# Validasyon loader: shuffle yok, drop_last yok (kaybı görmek için tam veri)
dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# --- 6. Model ve optimizasyon nesneleri tanımla ---

# MetricAE_MNA modeli (CIN: kanal sayısı, LSEQ: pencere uzunluğu, LATENT_DIM: latent boyutu, OFFS: komşu ofsetler)
model = MetricAE_MNA(CIN, LSEQ, LATENT_DIM, OFFS).to(DEVICE)

# Adam optimizasyon algoritması
opt = torch.optim.Adam(model.parameters(), lr=LR)

# Kayıp fonksiyonu: Ortalama Kare Hata (MSE)
mse = nn.MSELoss()


## 8. Eğitim döngüsü (MSE_center + β*MSE_neighbors + λ*NT-Xent)

- Erken durdurma (PATIENCE) ile en iyi modeli kaydeder.


In [22]:
from torch.serialization import add_safe_globals
add_safe_globals([np.core.multiarray._reconstruct])  # PyTorch 2.6 güvenlik güncellemesi için gerekli izin

# En iyi validasyon kaybını (loss) takip etmek için değişkenler
best_val = float("inf")   # Başlangıçta sonsuz (inf) atanır ki ilk değer daha küçük olsun
pat = 0                   # Erken durdurma (early stopping) için sabır (patience) sayacı
BEST = os.path.join(OUTDIR, "best_mna.pt")  # En iyi modelin kaydedileceği dosya yolu

# ---- EĞİTİM DÖNGÜSÜ ---- #
for epoch in range(1, EPOCHS+1):
    model.train()  # Modeli eğitim moduna al
    tr_c = tr_n = tr_con = tr_tot = 0.0  # Eğitimde kayıpları toplamak için değişkenler
    n = 0                                # Toplam örnek sayısı

    # Eğitim veri kümesi üzerinden minibatch döngüsü
    for x_clean, x1, x2, neigh, mask in dl_tr:
        # Verileri GPU/CPU’ya gönder
        x_clean = x_clean.to(DEVICE)
        x1 = x1.to(DEVICE); x2 = x2.to(DEVICE)
        neigh = neigh.to(DEVICE); mask = mask.to(DEVICE)

        opt.zero_grad()  # Önceki gradyanları temizle

        # ---- MODELİN ÇIKIŞLARI ---- #
        # y0: merkez pencere rekonstrüksiyonu
        # yN: komşu pencerelerin tahminleri (dict formatında)
        # zc: latent (kodlama)
        # pc: projeksiyon başı (contrastive öğrenme için)
        y0, yN, zc, pc = model(x_clean)

        # --- 1. Merkez pencere MSE kaybı ---
        loss_c = mse(y0, x_clean)

        # --- 2. Komşu pencereler kaybı (maskeli MSE ortalaması) ---
        loss_n, msum = 0.0, 0.0
        for j, d in enumerate(OFFS):           # OFFS: komşu kaymaları listesi
            pred = yN[str(d)]                  # İlgili komşunun model tahmini
            tgt  = neigh[:, j]                 # Gerçek komşu pencere
            m    = mask[:, j]                  # Geçerli frame maskesi
            l = ((pred - tgt)**2).mean(dim=(1,2))  # Her örnek için MSE (B,)
            loss_n += (l * m).sum()            # Maskelenmiş kayıpları topla
            msum   += m.sum()                  # Toplam geçerli frame sayısı
        loss_n = loss_n / (msum + 1e-8)        # Ortalama komşu kaybı

        # --- 3. Contrastive (NT-Xent) kaybı ---
        # encode: girdiyi latent vektöre çevirir
        # project: latent vektör üzerine ek bir dönüşüm (g(z))
        z1 = model.encode(x1); p1 = model.project(z1)
        z2 = model.encode(x2); p2 = model.project(z2)
        loss_con = nt_xent(p1, p2, temp=TEMP)  # Normalleştirilmiş sıcaklık skalalı kayıp

        # --- Toplam kayıp ---
        loss = loss_c + BETA_MNA*loss_n + LAMBDA_CONTR*loss_con
        loss.backward()  # Geri yayılım
        opt.step()       # Ağırlıkları güncelle

        # İstatistikler (epoch ortalaması için)
        bs = x_clean.size(0)         # Batch boyutu
        tr_c   += loss_c.item()*bs   # Merkez kaybı toplamı
        tr_n   += loss_n.item()*bs   # Komşu kaybı toplamı
        tr_con += loss_con.item()*bs # Contrastive kayıp toplamı
        tr_tot += loss.item()*bs     # Toplam kayıp
        n += bs

    # Eğitim ortalama kayıpları
    tr_c/=n; tr_n/=n; tr_con/=n; tr_tot/=n

    # ---------------- VALIDASYON ---------------- #
    model.eval()  # Modeli validasyon moduna al
    va_c = va_n = va_con = 0.0; nv = 0
    with torch.no_grad():  # Gradyan hesaplama kapalı
        for x_clean, x1, x2, neigh, mask in dl_va:
            x_clean = x_clean.to(DEVICE)
            x1 = x1.to(DEVICE); x2 = x2.to(DEVICE)
            neigh = neigh.to(DEVICE); mask = mask.to(DEVICE)

            # Validasyon için ileri besleme
            y0, yN, zc, pc = model(x_clean)
            lc = mse(y0, x_clean)  # Merkez kayıp

            # Komşu kaybı (maskeli ortalama)
            ln, msum = 0.0, 0.0
            for j, d in enumerate(OFFS):
                pred = yN[str(d)]
                tgt  = neigh[:, j]
                m    = mask[:, j]
                l = ((pred - tgt)**2).mean(dim=(1,2))
                ln += (l * m).sum(); msum += m.sum()
            ln = ln / (msum + 1e-8)

            # Contrastive kayıp
            z1 = model.encode(x1); p1 = model.project(z1)
            z2 = model.encode(x2); p2 = model.project(z2)
            lcon = nt_xent(p1, p2, temp=TEMP)

            # İstatistikleri topla
            bs = x_clean.size(0)
            va_c   += lc.item()*bs
            va_n   += ln.item()*bs
            va_con += lcon.item()*bs
            nv += bs

    # Validasyon ortalama kayıpları
    va_c/=nv; va_n/=nv; va_con/=nv
    va_tot = va_c + BETA_MNA*va_n + LAMBDA_CONTR*va_con

    # Epoch sonuçlarını yazdır
    print(f"[{epoch:03d}] TR tot {tr_tot:.4f} | c {tr_c:.4f} | n {tr_n:.4f} | con {tr_con:.4f} "
          f"|| VA tot {va_tot:.4f} | c {va_c:.4f} | n {va_n:.4f} | con {va_con:.4f}")

    # ---- En iyi modeli kaydetme ---- #
    if va_tot < best_val - 1e-5:  # Validasyon daha iyi ise
        best_val = va_tot; pat = 0
        torch.save({
            "state_dict": model.state_dict(),               # Model ağırlıkları
            "stats_mean": torch.from_numpy(ds_tr.get_stats()[0]).float(), # Normalizasyon ortalaması
            "stats_std":  torch.from_numpy(ds_tr.get_stats()[1]).float(), # Normalizasyon std
            "cin": CIN, "lseq": LSEQ, "latent": LATENT_DIM, # Model parametreleri
            "offs": OFFS, "beta": BETA_MNA, "lambda": LAMBDA_CONTR
        }, BEST)
    else:
        # Erken durdurma kontrolü
        pat += 1
        if pat >= PATIENCE:
            print("Erken durdurma."); break

print("Best val:", best_val, " | saved ->", BEST)


[001] TR tot 4.0358 | c 1.0109 | n 1.0532 | con 5.4178 || VA tot 3.6902 | c 0.8914 | n 0.9237 | con 5.0433
[002] TR tot 3.7010 | c 0.9708 | n 1.0284 | con 4.8434 || VA tot 3.4415 | c 0.8654 | n 0.9093 | con 4.6065
[003] TR tot 3.5671 | c 0.9728 | n 1.0434 | con 4.5626 || VA tot 3.3312 | c 0.8244 | n 0.8885 | con 4.4804
[004] TR tot 3.4301 | c 0.8949 | n 0.9951 | con 4.4734 || VA tot 3.2460 | c 0.7748 | n 0.8590 | con 4.4270
[005] TR tot 3.3384 | c 0.8251 | n 0.9662 | con 4.4467 || VA tot 3.1941 | c 0.7282 | n 0.8253 | con 4.4365
[006] TR tot 3.3404 | c 0.8445 | n 0.9537 | con 4.4197 || VA tot 3.1411 | c 0.7104 | n 0.8081 | con 4.3767
[007] TR tot 3.2547 | c 0.7814 | n 0.9196 | con 4.3950 || VA tot 3.1177 | c 0.7003 | n 0.7978 | con 4.3561
[008] TR tot 3.2221 | c 0.7780 | n 0.9020 | con 4.3471 || VA tot 3.0755 | c 0.6904 | n 0.7907 | con 4.2958
[009] TR tot 3.1594 | c 0.7482 | n 0.8754 | con 4.2972 || VA tot 3.0434 | c 0.6834 | n 0.7835 | con 4.2500
[010] TR tot 3.1526 | c 0.7687 | n 0.

## 9. Başarı yüzdesi ve latent çıkarımı
Aşağıdaki hücreleri istersen çalıştır. (Sunum çıktısı için faydalı.)


In [23]:

import numpy as np, torch
from torch.serialization import add_safe_globals

# PyTorch 2.6 güvenlik kısıtlaması için gerekli bir satır (numpy kayıt yapısına erişim)
add_safe_globals([np.core.multiarray._reconstruct])


# ✅ Autoencoder Başarı Yüzdesi Fonksiyonu
def success_percent(dloader, tau=None, device=DEVICE, mdl=model):
    """
    Verilen bir DataLoader üzerinden Autoencoder modelinin yeniden yapılandırma başarısını hesaplar.

    Girdi:
        dloader : DataLoader → verinin batch batch geldiği nesne
        tau     : float (isteğe bağlı) → başarı eşiği (MSE altında olursa başarılı sayılır)
        device  : PyTorch için kullanılacak donanım (CPU/GPU)
        mdl     : Kullanılacak model (varsayılan: global model)
    
    Dönüş:
        succ : yüzde başarı (0–100)
        tau  : kullanılan eşik değeri
    """
    errs = []  # Tüm MSE hataları burada toplanacak
    mdl.eval()
    with torch.no_grad():
        for batch in dloader:
            x_clean = batch[0].to(device)        # Giriş verisi (temiz)
            y0 = mdl(x_clean)[0]                 # Sadece merkez rekonstrüksiyon alınır
            se = (y0 - x_clean).pow(2).mean(dim=(1,2))  # Her pencere için MSE hesaplanır
            errs.append(se.cpu().numpy())        # NumPy'ya dönüştürüp listeye ekle
    errs = np.concatenate(errs)

    # Eğer dışarıdan eşik verilmediyse, 80. yüzdelik değer otomatik alınır (val için genelde bu yapılır)
    if tau is None:
        tau = float(np.quantile(errs, 0.80))     # Dinamik eşik

    # Belirlenen eşik altındaki örneklerin oranı → başarı
    succ = float((errs < tau).mean() * 100.0)
    return succ, tau


#  Latent Vektör Çıkarımı Fonksiyonu
@torch.no_grad()
def extract_latents(dloader, use_projection=True, device=DEVICE, mdl=model):
    """
    Girdi batch'leri için modelin encoder (veya projection head) çıkışlarını (latent vektör) çıkarır.

    Girdi:
        dloader        : DataLoader → hangi veriden latent çıkarılacak?
        use_projection : bool → g(z) mi (SimCLR projection) yoksa z (raw encoder) mi alınacak?
        device         : CPU / GPU
        mdl            : Kullanılan model

    Dönüş:
        Z : (N, D) boyutunda NumPy dizisi (L2 normalize edilmiş latentler)
    """
    Z = []  # Tüm latent vektörler burada birikecek
    mdl.eval()
    for batch in dloader:
        x_clean = batch[0].to(device)
        z = mdl.encode(x_clean)           # (B,D) encoder çıkışı
        if use_projection:
            z = mdl.project(z)            # projection head varsa kullan (SimCLR)
        z = torch.nn.functional.normalize(z, dim=1)  # vektörleri birim uzunluğa getir (cosine için önemli)
        Z.append(z.cpu().numpy())
    return np.concatenate(Z, axis=0)


#  Eğitim ve Validasyon için Başarı Oranlarını Hesapla
val_succ, tau = success_percent(dl_va, tau=None)   # Val setinden eşik ve başarı al
tr_succ,  _   = success_percent(dl_tr, tau=tau)     # Eğitim seti için aynı eşikle başarı al

# Başarıları yazdır
print(f"Başarı (%% MSE < {tau:.6f}) -> Train: {tr_succ:.2f}%% | Val: {val_succ:.2f}%%")





Başarı (%% MSE < 0.642613) -> Train: 80.66%% | Val: 79.69%%


## 10. Test  için başarı hesabı

Amaç: Eğitim/validasyon bittikten sonra, kaydedilmiş en iyi model (best_mna.pt) ile test videolarındaki pencerelerin rekonstrüksiyon hatasına bakmak.

Pencerenin kendisini ne kadar iyi yeniden kuruyoruz?

* AE pencereyi ne kadar iyi çizebiliyor? bunun testi :

In [24]:

import os, numpy as np, torch
from torch.utils.data import DataLoader
from torch.serialization import add_safe_globals

# --- PyTorch 2.6 güvenliği: bazı NumPy objelerinin serileştirilmesine izin ver ---
add_safe_globals([np.core.multiarray._reconstruct])

# 1) En iyi modeli ve train istatistiklerini yükle
BEST = os.path.join(OUTDIR, "best_mna.pt")          # Eğitimde kaydedilen en iyi modelin yolu
ckpt = torch.load(BEST, map_location=DEVICE, weights_only=False)  # Checkpoint'i cihazda aç

# Checkpoint’ten model konfigürasyonunu oku
cin   = int(ckpt["cin"])      # Girdi kanal sayısı (C)
lseq  = int(ckpt["lseq"])     # Pencere uzunluğu (L)
ldim  = int(ckpt["latent"])   # Latent boyutu
offs  = list(map(int, ckpt["offs"]))  # Komşu ofset listesi (MNA kolu)

# Model iskeletini oluştur ve ağırlıkları yükle
model = MetricAE_MNA(cin, lseq, ldim, offs).to(DEVICE)
model.load_state_dict(ckpt["state_dict"])  # Ağırlıklar
model.eval()                               # Değerlendirme moduna al (dropout/bn dursun)

# Train verisinden gelen z-score istatistikleri (normalize için ortalama-std)
stats = (ckpt["stats_mean"].numpy(), ckpt["stats_std"].numpy())

# 2) Val setini yeniden kur (τ eşiğini val’dan seçeceğiz)
all_files = list_npy(DIR_TR)                            # Eğitim klasöründeki tüm npy dosyaları
n_val = max(1, int(len(all_files)*VAL_SPLIT))           # Val ayırma sayısı (ör: %20)
val_files = all_files[-n_val:]                          # Son n dosyayı val olarak kullan

# Val veri kümesi: fit_stats=False => eğitimin istatistiklerini kullan (stats=…)
ds_val = PoseWindowDataset(val_files, WINDOW, STRIDE, fit_stats=False, stats=stats)
dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# 3) Test setini kur (DIR_TE içindeki npy’ler)
test_files = list_npy(DIR_TE)
assert len(test_files) > 0, "DIR_TE içinde test .npy bulunamadı."
ds_te = PoseWindowDataset(test_files, WINDOW, STRIDE, fit_stats=False, stats=stats)
dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# 4) Başarı fonksiyonu
# - Modelin forward'ı 4 değer döndürdüğü için sadece 1.'yi (merkez rekonstrüksiyon y0) alıyoruz.
# - Her pencere için MSE hesaplıyoruz; tau (eşik) verilmişse onu, verilmemişse val’daki 0.80 persentili kullanıyoruz.
def success_percent(dloader, tau=None):
    errs = []  # Her pencerenin MSE hatalarını toplayacağımız liste
    with torch.no_grad():
        for batch in dloader:
            x_clean = batch[0].to(DEVICE)                  # (B, C, L) merkez pencere
            y0 = model(x_clean)[0]                         # Model çıktılarının ilki: merkez rekonstrüksiyonu
            se = (y0 - x_clean).pow(2).mean(dim=(1,2))     # Her örnek için ortalama MSE (B,)
            errs.append(se.cpu().numpy())                  # CPU’ya al ve NumPy’a çevir

    # Tüm mini-batch’leri tek diziye birleştir
    errs = np.concatenate(errs)

    # Eşik (tau) yoksa, val seti için 80. yüzdelik (quantile) seç (anormallikleri dışarıda bırakmak için)
    if tau is None:
        tau = float(np.quantile(errs, 0.80))               # Val hatalarının %80’i bu değerin altında
    succ = float((errs < tau).mean() * 100.0)              # Yüzde başarı: MSE < tau olan pencerelerin oranı
    return succ, tau, errs

# 5) τ’yi val’dan al, aynı eşiği test’e uygula
val_succ, tau, _ = success_percent(dl_val, tau=None)       # Val’dan tau türet
test_succ, _, _  = success_percent(dl_te,  tau=tau)        # Aynı tau ile test başarısı

# Özet yazdır
print(f"[VAL] Başarı (% MSE < {tau:.6f}) = {val_succ:.2f}% | pencere: {len(ds_val)}")
print(f"[TEST] Başarı (% MSE < {tau:.6f}) = {test_succ:.2f}% | pencere: {len(ds_te)}")

# 6) Video-bazlı test başarısı
# - Her test dosyası için ayrı DataLoader kurup başarıyı ölçer.
# - Boş pencere çıkaran dosya varsa (çok kısa video vs.) 0% ve pencere=0 yazar.
def per_file_success(files, tau):
    rows = []
    for fp in files:
        ds_one = PoseWindowDataset([fp], WINDOW, STRIDE, fit_stats=False, stats=stats)
        if len(ds_one) == 0:
            rows.append((os.path.basename(fp), 0.0, 0))
            continue
        dl_one = DataLoader(ds_one, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
        succ, _, _ = success_percent(dl_one, tau=tau)      # Aynı tau ile tek dosya başarısı
        rows.append((os.path.basename(fp), succ, len(ds_one)))
    return rows

# Tabloyu üret ve yazdır
rows = per_file_success(test_files, tau)
print("\n== TEST video-bazlı başarılar ==")
for name, succ, nwin in rows:
    print(f"{name:>20s} : {succ:6.2f}%  (pencere={nwin})")


[VAL] Başarı (% MSE < 0.634003) = 79.69% | pencere: 128
[TEST] Başarı (% MSE < 0.634003) = 77.78% | pencere: 45

== TEST video-bazlı başarılar ==
          61_ang.npy :  58.33%  (pencere=12)
          62_ang.npy : 100.00%  (pencere=12)
          63_ang.npy :  54.55%  (pencere=11)
          64_ang.npy : 100.00%  (pencere=10)


## 11. Latent benzerlik başarısı (SimCLR’e uygun)
Amaç: “AE iyi çizdi mi?” yerine “encoder benzer paternleri yakın mı koymuş?” sorusunu yanıtlamak.

Encoder’ın, aynı pencerenin iki görünümünü (x1, x2) latent’te ne kadar yakınladığına baktık .

In [25]:
import torch, numpy as np, os
from torch.utils.data import DataLoader
from torch.serialization import add_safe_globals
add_safe_globals([np.core.multiarray._reconstruct])

# 1) En iyi modeli ve istatistikleri yükle
BEST = os.path.join(OUTDIR, "best_mna.pt")
ckpt = torch.load(BEST, map_location=DEVICE, weights_only=False)

cin   = int(ckpt["cin"]); lseq  = int(ckpt["lseq"])
ldim  = int(ckpt["latent"]); offs = list(map(int, ckpt["offs"]))
stats = (ckpt["stats_mean"].numpy(), ckpt["stats_std"].numpy())

model = MetricAE_MNA(cin, lseq, ldim, offs).to(DEVICE)
model.load_state_dict(ckpt["state_dict"]); model.eval()

# 2) Train/Val/Test dataset ve loader'ları (aynı z-score ile)
all_files = list_npy(DIR_TR)
n_val = max(1, int(len(all_files)*VAL_SPLIT))
val_files = all_files[-n_val:]; train_files = all_files[:-n_val]
test_files = list_npy(DIR_TE)
assert len(test_files) > 0, "DIR_TE içinde test .npy bulunamadı."

ds_tr = PoseWindowDataset(train_files, WINDOW, STRIDE, fit_stats=False, stats=stats)
ds_va = PoseWindowDataset(val_files,   WINDOW, STRIDE, fit_stats=False, stats=stats)
ds_te = PoseWindowDataset(test_files,  WINDOW, STRIDE, fit_stats=False, stats=stats)

dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# 3) Yardımcılar: embed et, cosine hesapla
@torch.no_grad()
def embed_loader(dloader, use_aug=False):
    Z = []
    for batch in dloader:
        x_clean, x1, x2 = batch[0].to(DEVICE), batch[1].to(DEVICE), batch[2].to(DEVICE)
        X = x1 if use_aug else x_clean
        z = model.encode(X)                         # (B, D)
        z = torch.nn.functional.normalize(z, dim=1) # L2-normalize
        Z.append(z.cpu().numpy())
    return np.concatenate(Z, axis=0)

@torch.no_grad()
def val_positive_cosines(dloader):
    cos = []
    for batch in dloader:
        x1, x2 = batch[1].to(DEVICE), batch[2].to(DEVICE)
        z1 = torch.nn.functional.normalize(model.encode(x1), dim=1)
        z2 = torch.nn.functional.normalize(model.encode(x2), dim=1)
        c  = (z1*z2).sum(dim=1)                    # (B,)
        cos.append(c.cpu().numpy())
    return np.concatenate(cos, axis=0)

# 4) Eşik: validasyon pozitif çiftlerinden τ_cos
val_pos = val_positive_cosines(dl_va)
tau_cos = float(np.quantile(val_pos, 0.10))  # en düşük %10'u da "benzer" saysın
print(f"τ_cos (val pozitif 10. persentil) = {tau_cos:.4f} | val_pos mean={val_pos.mean():.4f}")

# 5) Train embed havuzu ve Test embed'leri
Z_tr = embed_loader(dl_tr, use_aug=False)   # (N_tr, D)
Z_te = embed_loader(dl_te, use_aug=False)   # (N_te, D)

# 6) Her test penceresi için train'e en yakın cosine
#    (cosine = z_test @ Z_tr^T ; max'ını al)
max_cos = []
bs = 2048
for i in range(0, len(Z_te), bs):
    z = Z_te[i:i+bs]                         # (b, D)
    sim = z @ Z_tr.T                         # (b, Ntr)
    max_cos.append(sim.max(axis=1))
max_cos = np.concatenate(max_cos, axis=0)

latent_succ = float((max_cos >= tau_cos).mean() * 100.0)
print(f"[TEST - Latent] Başarı (max cosine ≥ τ_cos) = {latent_succ:.2f}% | pencere: {len(Z_te)}")

# 7) Video-bazlı latent başarı
def per_file_latent_success(files):
    rows = []
    start = 0
    # pencere uzunlukları dosya bazlı hesaplanıyor
    lengths = []
    for fp in files:
        ds_one = PoseWindowDataset([fp], WINDOW, STRIDE, fit_stats=False, stats=stats)
        lengths.append(len(ds_one))
    idx = 0
    for fp, n in zip(files, lengths):
        if n == 0:
            rows.append((os.path.basename(fp), 0.0, 0))
            continue
        z = Z_te[idx:idx+n]                   # o videonun pencereleri
        sim = z @ Z_tr.T
        succ = float((sim.max(axis=1) >= tau_cos).mean() * 100.0)
        rows.append((os.path.basename(fp), succ, n))
        idx += n
    return rows

rows = per_file_latent_success(test_files)
print("\n== TEST video-bazlı latent başarılar ==")
for name, succ, nwin in rows:
    print(f"{name:>20s} : {succ:6.2f}%  (pencere={nwin})")


τ_cos (val pozitif 10. persentil) = 0.9922 | val_pos mean=0.9954
[TEST - Latent] Başarı (max cosine ≥ τ_cos) = 51.11% | pencere: 45

== TEST video-bazlı latent başarılar ==
          61_ang.npy :  75.00%  (pencere=12)
          62_ang.npy :  66.67%  (pencere=12)
          63_ang.npy :  36.36%  (pencere=11)
          64_ang.npy :  20.00%  (pencere=10)


## 12. Latent benzerlik: KNN tabanlı eşiğe göre test başarısı

* “Sorgu penceresi train’de bir benzer bulabiliyor mu?”

* (%84.44)? Eşik aynı protokolden kalibre edildiği için adil; SimCLR’lı encoder gerçekten benzer paternleri yakınlaştırmış.

In [26]:
# --- Latent'leri çıkar (z veya p seçilebilir) ---
@torch.no_grad()
def embed_loader(dloader, use_projection=False):  # <<< default: z
    Z = []
    model.eval()
    for batch in dloader:
        x = batch[0].to(DEVICE)
        z = model.encode(x)              # z
        if use_projection:
            z = model.project(z)         # p = g(z)
        z = torch.nn.functional.normalize(z, dim=1)
        Z.append(z.cpu().numpy())
    return np.concatenate(Z, axis=0)

# --- KNN benzerlik eşiği (val->train) ve test başarısı ---
def max_cos_to_pool(Zq, Zpool, bs=2048):
    out=[]
    for i in range(0, len(Zq), bs):
        z = Zq[i:i+bs]
        out.append(z @ Zpool.T)
    return np.max(np.concatenate(out,axis=0), axis=1)

Z_tr = embed_loader(dl_tr, use_projection=False)  # z-uzayı
Z_va = embed_loader(dl_va, use_projection=False)
Z_te = embed_loader(dl_te, use_projection=False)

val2tr = max_cos_to_pool(Z_va, Z_tr)
# kalibrasyon: val'de %90 geri çağırma için 10. persentil
tau_knn = float(np.quantile(val2tr, 0.10))
te2tr   = max_cos_to_pool(Z_te, Z_tr)
succ_knn = float((te2tr >= tau_knn).mean()*100.0)

print(f"z-uzayı: τ_knn={tau_knn:.4f} | val→tr mean={val2tr.mean():.4f}")
print(f"[TEST - Latent KNN (z)] Başarı = {succ_knn:.2f}% | pencere={len(Z_te)}")


z-uzayı: τ_knn=0.9859 | val→tr mean=0.9929
[TEST - Latent KNN (z)] Başarı = 73.33% | pencere=45


In [27]:
qs = [0.05, 0.10, 0.20, 0.25, 0.30]
print("\n== z-uzayı KNN başarı (farklı persentiller) ==")
for q in qs:
    tauq = float(np.quantile(val2tr, q))
    succ = float((te2tr >= tauq).mean()*100.0)
    print(f"q={q:>4.2f}  τ={tauq:.4f}  ->  TEST={succ:5.2f}%")



== z-uzayı KNN başarı (farklı persentiller) ==
q=0.05  τ=0.9798  ->  TEST=80.00%
q=0.10  τ=0.9859  ->  TEST=73.33%
q=0.20  τ=0.9907  ->  TEST=64.44%
q=0.25  τ=0.9915  ->  TEST=53.33%
q=0.30  τ=0.9920  ->  TEST=51.11%
